# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'ratchev2'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here
d = load("p1a_extinction")
X = d['airmass'].to_numpy()
m = d['zeropoint_mag'].to_numpy()
m_sigma = d['sigma_quoted_mag'].to_numpy()

A = np.column_stack((np.ones(len(X)), X))
Sinv = np.diag(1.0 / m_sigma**2)
cov_wls = np.linalg.inv(A.T @ Sinv @ A)
theta_wls = cov_wls @ A.T @ Sinv @ m
m0_wls, k_wls = theta_wls
sigma_m0_wls, sigma_k_wls = np.sqrt(np.diag(cov_wls))

print("WLS")
print("k: ", round(k_wls, 3), "+/-", round(sigma_k_wls, 4), "mag/airmass")
print("m0: ", round(m0_wls, 3), "+/-", round(sigma_m0_wls, 3), "mag")
print("")

p_polyfit, cov_polyfit = np.polyfit(X, m, 1, w= 1 / m_sigma, cov=True)
k_polyfit, m0_polyfit = p_polyfit
sigma_k_polyfit, sigma_m0_polyfit = np.sqrt(np.diag(cov_polyfit))

print("Numpy Polyfit")
print("k: ", round(k_polyfit, 3), "+/-", f"{sigma_k_polyfit:.3f}", "mag/airmass")
print("m0: ", round(m0_polyfit, 3), "+/-", round(sigma_m0_polyfit, 3), "mag")
print("")

from scipy.stats import chi2

residuals = m - A @ theta_wls
chi2 = np.sum((residuals / m_sigma)**2)
dof = len(X) - 2
chi2_red = chi2 / dof

print("Chi Squared:", round(chi2, 3))
print("DOF:", round(dof, 3))
print("Reduced Chi Squared:", round(chi2_red, 3))
print(f"Chi squared probability: {stats.chi2.sf(chi2, df = dof):.3g}")
print("")

# Adjusting error bars due to huge chi2 (here I am assuming model is correct)
sigma_k_new = sigma_k_wls * np.sqrt(chi2_red)
print("Adjusted k error bar:")
print("k: ", round(k_polyfit, 3), "+/-", f"{sigma_k_new:.3f}", "mag/airmass")

xs = np.linspace(1, 2.3, 1000)
plt.errorbar(X, m, yerr = m_sigma, fmt='o', label = "Data")
plt.plot(xs, k_wls * xs + m0_wls, label = "Best fit")
plt.legend()
plt.xlabel("X [airmass]")
plt.ylabel("m [mag]")
plt.show()

**ANSWER:**

Polyfit assumes that the weights we are giving (which is the inverse of the quoted sigma values) are relative rather than being absolute. This means that it scales up the parameter covariance matrix (and thus the uncertainties) by the square root of the reduced chi 2 of the fit such that chi_2 divided by the degrees of freedom goes to 1. This is why it gets a larger number. We can see from part 3 that without doing this scaling we have a reduced chi squared of 12.985, which has a probability of 3e-29 of being this large. If the linear model is correct and the uncertainties are correct as stated, this result is clearly incredibly unlikely. Therefore, I scaled up the error bars as polyfit does in order to get a chi2 of one. This assumes that the model is correct and that there are no outlier problems. While these are potentially dangerous assumptions to make, as a high chi2 usually warns that something is wrong and that can many times not be related to the scaling of the uncertainties, a plot of the data shows no clear outliers and instead what appear to be systematically small error bars. Thus I report the scaled value of k = 0.156 +/- 0.040 mag/airmass.

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here
d = load("p1b_centroid")

N = d['n_photons'].to_numpy()
psf_sigma = d['psf_sigma_arcsec'].to_numpy()
quoted_sigma = d['quoted_sigma_x0_arcsec'].to_numpy()

cr = psf_sigma[0] / np.sqrt(N[0]) # it is simply this because for a Gaussian of a certain width the fisher information is just 1/sigma^2_PSF for one photon and for N you just multiply this by N. Now you invert for CR bound on variance. For error bar we then square root.
print("CR bound:", round(cr, 7), "arcsec")
print("Error quoted:", quoted_sigma[0], "arcsec")

**ANSWER:**

The quoted error is smaller than the CR bound which means that it cannot be the correct error of an unbiased estimator under the Gaussian model. The classmate's number must therefore mean that their uncertainty is calculated incorrectly. Two mistakes that could have produced an error bar this small are confusing the scaling of the variance with that of the error bar and so dividing by N instead of the square root of N, or by failing to remove background photon counts such that N is overestimated.

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here
d = load("p1c_profile")
depth_ppt = d["depth_ppt"].to_numpy()
minus2lnL = d["minus2lnL"].to_numpy()

minimum_idx = np.argmin(minus2lnL)
depth_ppt_min = depth_ppt[minimum_idx]
min_value = minus2lnL[minimum_idx]
min_points = np.abs(depth_ppt - depth_ppt_min) < 0.15 
a, b, c = np.polyfit(depth_ppt[min_points] - depth_ppt_min, minus2lnL[min_points], 2)

curvature = 2 * a
sigma_curvature = np.sqrt(2 / curvature)
sigma_classmate = 1 / np.sqrt(curvature)

print("Best depth from curvature:", round(depth_ppt_min, 2), "ppt")
print("Correct sigma:", round(sigma_curvature, 2), "ppt")
print("Sigma given by classmate:", round(sigma_classmate, 2), "ppt")
print("Factor:", round(sigma_curvature / sigma_classmate, 2))

depth_left = depth_ppt[np.where(minus2lnL[:minimum_idx] >= min_value  + 1)[0][-1]]
depth_right = depth_ppt[minimum_idx + 1 + np.where(minus2lnL[minimum_idx+1:] >= min_value + 1)[0][0]]
print("")
print("delta(-2lnL) = 1 interval:")
print("Left crossing difference:", round(depth_ppt_min - depth_left, 2), "ppt")
print("Right crossing difference:", round(depth_right - depth_ppt_min, 2), "ppt")

plt.scatter(depth_ppt, minus2lnL - min_value, marker = "o")
plt.axhline(1, color="red", linestyle="--", label="Delta(-2lnL)= 1")
plt.xlabel("Transit depth [ppt]")
plt.ylabel("Delta(-2 ln L)")
plt.legend()
plt.show()

np.random.seed(100)
N = 10000

sim_depths = np.random.normal(depth_ppt_min, sigma_curvature, N)
cover_num = np.abs(sim_depths - depth_ppt_min) <= sigma_classmate

print("Actual coverage:", np.sum(cover_num) / N)

**ANSWER:**

As can be seen, the classmate's error bar is off by a factor of square root 2 and this is because the curvature is given by 2a where a is the quadratic coefficient and the correct uncertainty is sigma = sqrt(2/curvature) rather than just sqrt(1/curvature). This accounts for this factor. The Delta(-2LnL) = 1 interval is the minima at 2.8 + 0.23 ppt and - 0.26 ppt and so this is reasonably symmetric and agrees well with the 0.23 ppt curvature answer, though not completely with the lower transit side. Since the classmate's uncertainty is smaller by a factor of sqrt(2). the interval will only go out to about 0.707 standard deviations. For Gaussian errors, this contains about 52% of measurements (versus what we want of about 68%), and this is exactly what we see in the simulated run.

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here
d = load("p1d_mass_samples")
M2_Msun = d['M2_Msun'].to_numpy()

mass16, mass84, median = np.percentile(M2_Msun, (16, 84, 50))
print("Mass of M2 =", round(median, 1), "+", round(mass84 - median, 1), "/ -", round(median - mass16, 1), "Msun")
print("Variance:", round(np.var(M2_Msun, ddof = 1), 3), "Msun^2")
print("84th percentile minus 16th:", round(mass84-mass16, 3), "Msun")
print("Standard deviation / square root of 8000:", round(np.std(M2_Msun, ddof = 1) / np.sqrt(8000), 3), "Msun")

plt.hist(M2_Msun, bins=100, density=True)
plt.axvline(mass16, color = 'red', label='16th percentile')
plt.axvline(median, color='black', label='Median')
plt.axvline(mass84, color='pink', label='84th percentile')
plt.xlabel("Mass of companion [Msun]")
plt.ylabel("Posterior density")
plt.legend()
plt.show()

**ANSWER:**

For (i), the quantity reported is not the uncertainty but rather the spread of the data squared, and therefore has wrong units of Msun^2. Taking the square root of this vaue is a decent estimate for the 1sigma error bar if the posterior distribution is not too skewed. For (ii), the quantity reported is the width of the 68% interval (not the one sided error bars we are looking for), which is in the correct units of Msun but is a factor of 2 bigger than the standard deviation for a Gaussian which is the 1sigma error bar for a symmetric distribution. For (iii), the quantity given is the error of the estimated sample mean while what we are looking for is the uncertainty in the companian mass itself. This does have the correct units but is an underestimate by a factor of sqrt(N).

Symmetric error bars are not appropriate for this posterior, as seen in the histogram above as well as in my computed 16th and 84th percentile error bars. While the distribution is not incredibly skewed, it is still skewed enough that the + and - error bars should be reported separately.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here
d = load("p1e_slope_intercept_samples")
slope = d["slope"].to_numpy()
intercept = d["intercept"].to_numpy()

marginal_err = np.std(slope, ddof = 1)
intercept_fixed = np.mean(intercept)
classmate_err = np.std(slope[np.abs(intercept - intercept_fixed) < 0.05 * np.std(intercept)], ddof = 1)

ratio = marginal_err / classmate_err 
cc = np.corrcoef(slope, intercept)[0, 1]

print("Marginal slope error:", round(marginal_err, 3))
print("Classmate slope error:", round(classmate_err, 3))
print("Ratio:", round(ratio, 3))
print("Correlation Coefficient:", round(cc, 3))

plt.scatter(intercept, slope)
plt.axhline(np.mean(slope), color="black", label="Mean slope")
plt.axhline(np.mean(slope) + classmate_err, color="red", label="Classmate error bar")
plt.axhline(np.mean(slope) - classmate_err, color="red")
plt.axhline(np.mean(slope) + marginal_err, color="orange", label="Marginal error bar")
plt.axhline(np.mean(slope) -marginal_err, color="orange")
plt.xlabel("Intercept")
plt.ylabel("Slope")
plt.legend()
plt.show()

**ANSWER:**

We can see that the slope error calculated by our classmate is less than the marginal result by a large factor. This ratio increases as the magnitude of the correlation coefficient increases. When the slope and intercept are uncorrelated, the ratio approaches one. Therefore, the classmate's number is the right one to quote when the intercept is either known very precisely or our model has it fixed rather than it being just another fitting parameter.

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here
import astropy
from astropy.cosmology import FlatLambdaCDM

d = load("p2_pantheon_sne")
mB = d['mB'].to_numpy()
mB_err = d['mB_err'].to_numpy()
x1 = d['x1'].to_numpy()
x1_err = d['x1_err'].to_numpy()
c = d['c'].to_numpy()
c_err = d['c_err'].to_numpy()
cov_mB_x1 = d['cov_mB_x1'].to_numpy()
cov_mB_c = d['cov_mB_c'].to_numpy()
cov_x1_c = d['cov_x1_c'].to_numpy()
z = d['z'].to_numpy()
host_logmass = d['host_logmass'].to_numpy()
host_logmass_err= d['host_logmass_err'].to_numpy()


mu = astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z).value
residuals = mB - mu

plt.scatter(x1, residuals)
plt.xlabel("Stretch x1 [dimensionless]")
plt.ylabel("Hubble residual [mag]")
plt.show()

plt.scatter(c, residuals)
plt.xlabel("Color [dimensionless]")
plt.ylabel("Hubble residual [mag]")
plt.show()

plt.scatter(host_logmass, residuals)
plt.xlabel("Log 10 (M_stellar/M_sun)")
plt.ylabel("Hubble residual [mag]")
plt.show()

plt.scatter(host_logmass, x1)
plt.ylabel("Stretch x1 [dimensionless]")
plt.xlabel("Log 10 (M_stellar/M_sun)")
plt.show()

**ANSWER:**

The trend of the residuals against x1 is negative, which is what the Tripp relation expects since the term with x1 is negative alpha. Meanwhile, the trend of the residuals against the color is positive, which is what the Tripp relation expects since the term with c is positive beta. The raw trend with host mass may not have the sign of gamma because the host mass can be correlated with the stretch and the color which cannot be separated visually from just the effect of the host mass and so we cannot isolate that single sign with the current plot. As shown in the final plot, the stretch does appear to grow larger with decreasing host galaxy mass, which means that the difference across host mass can be explained at least partially by the stretch term rather than gamma. The contaminants have their mB fainter by at least a magnitude (aka mB increased by one or more) so their residuals will be unusually positive. It is hard to fully tell if a certain point is a real outlier, though the most likely candidates for the contaminants are the three points on the top left of the residual vs. color plot. The trend in this plot is much more compact than the others and these points stand out. Since both x1 and c are dimensionless, and alpha and beta multiply these quantities, their units are just magnitudes as they must agree with the Tripp relation being expressed in magnitudes.

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here
Cov = np.zeros((len(mB), 3, 3))
Cov[:, 0, 0] = mB_err**2
Cov[:, 1, 1] = x1_err**2
Cov[:, 2, 2] = c_err**2

Cov[:, 0, 1] = Cov[:, 1, 0] = cov_mB_x1
Cov[:, 0, 2] = Cov[:, 2, 0] = cov_mB_c
Cov[:, 1, 2] = Cov[:, 2, 1] = cov_x1_c

def log_posterior(theta, mu = mu, x1 = x1, c = c, mB = mB, host_logmass = host_logmass, mB_err = mB_err, Cov = Cov, z = z):
    M, alpha, beta, gamma, log_sigma_int = theta
    # Priors
    if not (-20 < M < -18):
        return -np.inf
    if not (0 < alpha < 1):
        return -np.inf
    if not (1 < beta < 6):
        return -np.inf
    if not (-0.3 < gamma < 0.3):
        return -np.inf
    sigma_int = np.exp(log_sigma_int)
    

    sigma_mu_pec = (5 / np.log(10)) * 250 / (299792 * z)
    Theta = (host_logmass > 10).astype(int)

    model_mB = (M + mu - alpha * x1 + beta * c + gamma * Theta)
    res = mB - model_mB

    v = np.array([1, alpha, -1 * beta])
    vT_C_v = np.zeros(len(mB))
    for i in range(len(mB)):
        vT_C_v[i] = v @ Cov[i] @ v
    total_var = (vT_C_v + sigma_int**2 + sigma_mu_pec**2)

    log_likelihood = -0.5 * np.sum(np.log(total_var) + res**2 / total_var)
    log_prior_sigma_int = np.log(2 / np.sqrt(2 * np.pi * 0.25**2)) - 0.5 * (sigma_int / 0.25)**2
    return log_likelihood + log_prior_sigma_int + log_sigma_int # I added this log_sigma to stop samplng bias due to probability density not being linear

def neg_log_posterior(theta, mu = mu, x1 = x1, c = c, mB = mB, host_logmass = host_logmass, mB_err = mB_err, Cov = Cov, z = z):
    return -log_posterior(theta, mu, x1, c, mB, host_logmass, mB_err, Cov, z)

theta0 = [-19, 0.2, 4, -0.1, np.log(0.2)]
res = minimize(neg_log_posterior, theta0, method = 'Nelder-Mead')
M_fit, alpha_fit, beta_fit, gamma_fit, log_sigma_int_fit = res.x
sigma_int_fit = np.exp(log_sigma_int_fit)

Theta = (host_logmass > 10).astype(int)
model_mB = (M_fit + mu - alpha_fit * x1 + beta_fit * c + gamma_fit * Theta)
residuals = mB - model_mB

plt.scatter(np.arange(len(mB)), residuals)
plt.xlabel("Supernova Number")
plt.ylabel("Residual [mag]")
plt.show()

idx = np.argsort(residuals)[::-1]
print("highest residual point:", idx[0], "residual (mag):", round(residuals[idx[0]], 3))
print("second highest residual point:", idx[1], "residual (mag):", round(residuals[idx[1]], 3))
print("third highest residual point:", idx[2], "residual (mag):", round(residuals[idx[2]], 3))

**ANSWER:**

The off diagonal terms are in our calculation of v^T_C_v because they account for any correlations between the measurement errors for m_B, x1 and c. We cannot just use their respective errors and pretend they are independent.

The priors I chose are as follows:

M: uniform between -18 and -20 mag. I chose this because this is a plausible range of magnitudes given my plots from the first part where the y axis mB - mu spanned these values and M represents the vertical intercept after correcting for x1,c, and host mass and that intercept judging from the plots should be in this range.

alpha: uniform between 0 and 1. I chose this because of the slight downward trend shown in residual versus x1 plot, where the relation containing -alpha * x1 means that alpha here will be some small positive slope so 0 to 1 is my range.

beta: uniform between 1 and 6. I chose this because of the stronger upward trend shown in the residual vs c plot, where the relation containing beta * c means that beta here will be some slope, and this slope clearly has a greater magnitude than the alpha slope and as such I upped the range.

gamma: uniform between -0.3 and 0.3. I chose this because it is unclear if the trend will be upward or downward even with the plot due to stretch and color varying with hostmass, however it is clearly a week trend and so a value close to 0 should be correct.

sigma_int: absolute value of gaussian with 0.25 standard deviation. I chose this because sigma_int needs to be nonnegative, and I cannot tell from the data a "cap" for sigma int however I expect it to be relatively small compared to the full magnitude range and so a Gaussian does that job for me.

I chose to sample ln sigma_int instead of just sigma_int because sigma_int must be positive and sampling ln sigma_int ensures that. I additionally decided to remove the contaminants. This was done by taking an initial guess of the parameters and using a Nelder-Mead minimization of the negative log posterior, and then plotting the residuals between the modelled mB and the recorded value. Three points stood out by a large margin compared to the rest of the data as seen in the plot above (with values close to 1 mag), and these were ruled the contaminants and removed. This cut comes at the cost of reducing the sample size and may introduce some selection bias and an underestimate of parameter uncertainties if this decision was not correctly justified.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here
import emcee
import corner
import arviz as az

contaminant_idx = [59, 46, 50]
mask = np.ones(len(mB), dtype = bool)
mask[contaminant_idx] = False

mB_kept = mB[mask]
mB_err_kept = mB_err[mask]
x1_kept = x1[mask]
c_kept = c[mask]
z_kept = z[mask]
mu_kept = mu[mask]
host_logmass_kept = host_logmass[mask]
Cov_kept = Cov[mask]

Theta_kept = (host_logmass_kept > 10).astype(int)
sigma_mu_pec_kept = (5 / np.log(10)) * 250 / (299792 * z_kept)
# FINDING MAP POINT
theta0 = [M_fit, alpha_fit, beta_fit, gamma_fit, log_sigma_int_fit]
map_res = minimize(neg_log_posterior, theta0, args = (mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept), method = 'Nelder-Mead')
theta_map = map_res.x
print("MAP parameter results:", np.round(theta_map, 3))
print("MAP sigma int:", np.round(np.exp(theta_map[4]), 3))

In [ ]:
# using Hessian to get widths of parameters which I use to set scale of chains 
h = 1e-4
n = len(theta_map)
hess = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        ei, ej = np.zeros(n), np.zeros(n)
        ei[i] = h
        ej[j] = h

        hess[i, j] = (
            neg_log_posterior(theta_map + ei + ej, mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept)
            - neg_log_posterior(theta_map + ei - ej, mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept)
            - neg_log_posterior(theta_map - ei + ej, mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept)
            + neg_log_posterior(theta_map - ei - ej, mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept)
        ) / (4 * h**2)
widths = np.sqrt(np.diag(np.linalg.inv(hess)))

rng = np.random.default_rng(3)

start_scale = 2 * widths

samplers = []

for k in range(4):
    theta_start = theta_map + start_scale * rng.normal(size=(16, 5))
    np.random.seed(10)
    sampler = emcee.EnsembleSampler(nwalkers = 16, ndim = 5, log_prob_fn = log_posterior, args=(mu_kept, x1_kept, c_kept, mB_kept,host_logmass_kept, mB_err_kept, Cov_kept, z_kept))
    sampler.run_mcmc(theta_start, 10000)
    samplers.append(sampler)

In [ ]:
# R and ESS for all parameters:
for k, sampler in enumerate(samplers):
    print("Sampler", k)
    tau = sampler.get_autocorr_time()
    print("Autocorrelation time", np.round(tau, 2))
    print("Steps / autocorrelation time", np.round(sampler.get_chain().shape[0] / tau.max(), 1))
    chain = sampler.get_chain(discard = int(3 * tau.max()), thin = max(1, int(tau.max() / 2)))
    az_chain = np.transpose(chain, (1, 0, 2))

    posterior = {
    "M": az_chain[:, :, 0],
    "alpha": az_chain[:, :, 1],
    "beta": az_chain[:, :, 2],
    "gamma": az_chain[:, :, 3],
    "log_sigma_int": az_chain[:, :, 4]}
    idata = az.from_dict({"posterior": posterior})
    print(az.summary(idata, var_names = ["M", "alpha", "beta", "gamma", "log_sigma_int"]))

In [ ]:
all_samples = []

for sampler in samplers:
    tau = sampler.get_autocorr_time()
    chain = sampler.get_chain(discard = int(3 * tau.max()), thin = max(1, int(tau.max() / 2)), flat = True)
    all_samples.append(chain)

samples = np.concatenate(all_samples, axis=0)
samples[:, 4] = np.exp(samples[:, 4]) # since sigma int is still in log 
corner.corner(samples, labels=["M", "alpha", "beta", "gamma", "sigma_int"], quantiles=[0.16, 0.50, 0.84], show_titles=True)
plt.show()

for i, var in enumerate(["M", "alpha", "beta", "gamma", "sigma_int"]):
    q16, q84, median = np.percentile(samples[:, i], [16, 84, 50])
    print(var, "=", round(median, 2), "+", round(q84 - median, 2), "/ -", round(median - q16, 2))

In [ ]:
M_fixed = theta_map[0]

def neg_log_posterior_M(theta):
    alpha, beta, gamma, log_sigma_int = theta
    return neg_log_posterior([M_fixed, alpha, beta, gamma, log_sigma_int], mu_kept, x1_kept, c_kept, mB_kept, host_logmass_kept, mB_err_kept, Cov_kept, z_kept)
res_M_fixed = minimize(neg_log_posterior_M, theta_map[1:], method = 'Nelder-Mead')
alpha_M_fixed, beta_M_fixed, gamma_M_fixed, log_sigma_int_M_fixed = res_M_fixed.x

print("Fixed M =", round(M_fixed, 2))
print("Gamma refitted value =", round(gamma_M_fixed, 3))

n = 4
h = 1e-4
hess_fixed = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = np.zeros(n), np.zeros(n)
        ei[i] = h
        ej[j] = h

        hess_fixed[i, j] = (
            neg_log_posterior_M([alpha_M_fixed, beta_M_fixed, gamma_M_fixed, log_sigma_int_M_fixed] + ei + ej)
            - neg_log_posterior_M([alpha_M_fixed, beta_M_fixed, gamma_M_fixed, log_sigma_int_M_fixed] + ei - ej)
            - neg_log_posterior_M([alpha_M_fixed, beta_M_fixed, gamma_M_fixed, log_sigma_int_M_fixed] - ei + ej)
            + neg_log_posterior_M([alpha_M_fixed, beta_M_fixed, gamma_M_fixed, log_sigma_int_M_fixed] - ei - ej)
        ) / (4 * h**2)

widths_M_fixed = np.sqrt(np.diag(np.linalg.inv(hess_fixed)))
sigma_gamma_M_fixed = widths_M_fixed[2]
print("")
print("Conditional sigma_gamma =", sigma_gamma_M_fixed)
q16_gamma, q84_gamma, gamma_median = np.percentile(samples[:, 3], [16, 84, 50])

print("Marginal gamma:")
print("median =", round(gamma_median, 3))
print("lower uncertainty =", round(gamma_median - q16_gamma, 3))
print("upper uncertainty =", round(q84_gamma - gamma_median, 3))

**ANSWER:**

The MAP point was found doing a minimization of the negative log posterior using the Nelder-Mead method. I then ran four emcee chains with 16 walker each where I got the initial locations of the walkers by using the inverse Hessian to get the local standard deviations of the parameters and then took 2 times the given values as the scale to be multiplied by a normal Gaussian distribution and added to the start value of the MAP point.

The rhat for each of my samplers for all parameters is <=1.01 and the ess_bulk and ess_tail are in the 2000-3000 range. I would have run for longer if I got an rhat greater than 1.01 or an ess value of less than 400.

The strongest correlation is between M and gamma, which are negatively correlated. This is because M shifts the magnitudes of all of the supernovas while gamma only shifts the ones that have a host galaxy with a high mass. This means that one going down can be compensated by an increase in the other. There is not as strong correlations for anything else, although there is a slight negative correlation between beta and sigma int, as well as alpha and gamma. This correlation between alpha and gamma is exactly what we saw in the first residual plots. The beta and sigma int negative correlation could be due to the fact that changing beta causes the model to better or worse account for color in a way that makes the residual scatter seem to cover a wider or smaller range. However, these are not as clear of a relationship as that between M and gamma. There is also a slight positive correlation between M and beta and M and alpha, which can be explained similarly to the M and gamma case where a higher beta or alpha can compensate a bit for a lower M, but this effect is not as great as that of gamma which directly looks at host galaxies.

We can see that fixing M reduces the uncertainty in gamma from about 0.019 to 0.0124. This is consistent with the strong correlation we can see in the corner plot because fixing M removes the uncertainty associated with this correlation.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here

def log_posterior_M0(theta):
    M, alpha, beta, log_sigma_int = theta
    gamma = 0
    # Priors
    if not (-20 < M < -18):
        return -np.inf
    if not (0 < alpha < 1):
        return -np.inf
    if not (1 < beta < 6):
        return -np.inf

    sigma_int = np.exp(log_sigma_int)
    model_mB = (M + mu_kept - alpha * x1_kept + beta * c_kept + gamma * Theta_kept)
    res = mB_kept - model_mB

    v = np.array([1, alpha, -1 * beta])
    vT_C_v = np.zeros(len(mB_kept))
    for i in range(len(mB_kept)):
        vT_C_v[i] = v @ Cov_kept[i] @ v
    total_var = (vT_C_v + sigma_int**2 + sigma_mu_pec_kept**2)

    log_likelihood = -0.5 * np.sum(np.log(total_var) + res**2 / total_var)
    log_prior_sigma_int = np.log(2 / np.sqrt(2 * np.pi * 0.25**2)) - 0.5 * (sigma_int / 0.25)**2
    return log_likelihood + log_prior_sigma_int + log_sigma_int
def neg_log_posterior_M0(theta):
    return -1 * log_posterior_M0(theta)
def log_posterior_M1(theta, gamma_prior_scale):
    M, alpha, beta, gamma, log_sigma_int = theta
    # Priors
    if not (-20 < M < -18):
        return -np.inf
    if not (0 < alpha < 1):
        return -np.inf
    if not (1 < beta < 6):
        return -np.inf
    if not (-gamma_prior_scale < gamma < gamma_prior_scale):
        return -np.inf

    sigma_int = np.exp(log_sigma_int)
    model_mB = (M + mu_kept - alpha * x1_kept + beta * c_kept + gamma * Theta_kept)
    res = mB_kept - model_mB

    v = np.array([1, alpha, -1 * beta])
    vT_C_v = np.zeros(len(mB_kept))
    for i in range(len(mB_kept)):
        vT_C_v[i] = v @ Cov_kept[i] @ v
    total_var = (vT_C_v + sigma_int**2 + sigma_mu_pec_kept**2)

    log_likelihood = -0.5 * np.sum(np.log(total_var) + res**2 / total_var)
    log_prior_sigma_int = np.log(2 / np.sqrt(2 * np.pi * 0.25**2)) - 0.5 * (sigma_int / 0.25)**2
    return log_likelihood + log_prior_sigma_int + log_sigma_int
def neg_log_posterior_M1(theta, gamma_prior_scale):
    return -1 * log_posterior_M1(theta, gamma_prior_scale)
def hessian(f, theta, h = 1e-4):
    n = len(theta)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n)
            ej = np.zeros(n)
            ei[i] = h
            ej[j] = h

            H[i, j] = (
                f(theta + ei + ej)
                - f(theta + ei - ej)
                - f(theta - ei + ej)
                + f(theta - ei - ej)
            ) / (4 * h**2)

    return H
def hessian1(f, theta, gamma_prior_scale, h = 1e-4):
    n = len(theta)
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n)
            ej = np.zeros(n)
            ei[i] = h
            ej[j] = h

            H[i, j] = (
                f(theta + ei + ej, gamma_prior_scale)
                - f(theta + ei - ej, gamma_prior_scale)
                - f(theta - ei + ej, gamma_prior_scale)
                + f(theta - ei - ej, gamma_prior_scale)
            ) / (4 * h**2)

    return H


theta0_M0 = np.array([theta_map[0], theta_map[1], theta_map[2], theta_map[4]])
theta0_M1 = np.array([theta_map[0], theta_map[1], theta_map[2], theta_map[3], theta_map[4]])

res_0 = minimize(neg_log_posterior_M0, theta0_M0, method="Nelder-Mead")
hessian_0 = hessian(neg_log_posterior_M0, res_0.x)
cov_0 = np.linalg.inv(hessian_0)
widths_0 = np.sqrt(np.diag(cov_0))

log_term_M0 = log_posterior_M0(res_0.x) + len(res_0.x) / 2 * np.log(2 * np.pi) + 0.5 * np.log(np.linalg.det(cov_0))

res_1_0_3 = minimize(neg_log_posterior_M1, theta0_M1, args = (0.3,), method = "Nelder-Mead")
res_1_0_6 = minimize(neg_log_posterior_M1, theta0_M1, args = (0.6,), method = "Nelder-Mead")

hessian_1_0_3 = hessian1(neg_log_posterior_M1, res_1_0_3.x, 0.3)
hessian_1_0_6 = hessian1(neg_log_posterior_M1, res_1_0_6.x, 0.6)

cov_1_0_3 = np.linalg.inv(hessian_1_0_3)
cov_1_0_6 = np.linalg.inv(hessian_1_0_6)
# here I add in a term for the gamma_prior_scale since its no longer constant
log_term_M1_3 = log_posterior_M1(res_1_0_3.x, 0.3) - np.log(2 * 0.3) + len(res_1_0_3.x) / 2 * np.log(2 * np.pi) + 0.5 * np.log(np.linalg.det(cov_1_0_3))
log_term_M1_6 = log_posterior_M1(res_1_0_6.x, 0.6) - np.log(2 * 0.6) + len(res_1_0_6.x) / 2 * np.log(2 * np.pi) + 0.5 * np.log(np.linalg.det(cov_1_0_6))

two_ln_B_0_3 = 2 * (log_term_M1_3 - log_term_M0)
two_ln_B_0_6 = 2 * (log_term_M1_6 - log_term_M0)

print("Gamma prior: uniform -0.3 to 0.3")
print("")
print("Gamma:", round(res_1_0_3.x[3], 3), "+/-", round(np.sqrt(np.diag(cov_1_0_3))[3], 3))
print("2lnB_1,0:", round(two_ln_B_0_3, 3))
print("")
print("Gamma prior: uniform -0.6 to 0.6")
print("")
print("Gamma:", round(res_1_0_6.x[3], 3), "+/-", round(np.sqrt(np.diag(cov_1_0_6))[3], 3))
print("2lnB_1,0:", round(two_ln_B_0_6, 3))

In [ ]:
def log_likelihood_M0(theta):
    M, alpha, beta, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)

    model_mB = M + mu_kept - alpha * x1_kept + beta * c_kept
    res = mB_kept - model_mB

    v = np.array([1, alpha, -beta])
    vT_C_v = np.zeros(len(mB_kept))
    for i in range(len(mB_kept)):
        vT_C_v[i] = v @ Cov_kept[i] @ v
    total_var = vT_C_v + sigma_int**2 + sigma_mu_pec_kept**2

    return -0.5 * np.sum(np.log(total_var) + res**2 / total_var)
def log_likelihood_M1(theta):
    M, alpha, beta, gamma, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)

    model_mB = (M + mu_kept - alpha * x1_kept + beta * c_kept + gamma * Theta_kept)
    res = mB_kept - model_mB

    v = np.array([1, alpha, -1 * beta])
    vT_C_v = np.zeros(len(mB_kept))
    for i in range(len(mB_kept)):
        vT_C_v[i] = v @ Cov_kept[i] @ v
    total_var = vT_C_v + sigma_int**2 + sigma_mu_pec_kept**2

    return -0.5 * np.sum(np.log(total_var) + res**2 / total_var)
def neg_log_likelihood_M0(theta):
    return -log_likelihood_M0(theta)
def neg_log_likelihood_M1(theta):
    return -log_likelihood_M1(theta)

# Maximum-likelihood fit for M0
fit_0 = minimize(neg_log_likelihood_M0, res_0.x, method="Nelder-Mead", bounds=[(-20, -18), (0, 1), (1, 6), (-15, 0)])
fit_1 = minimize(neg_log_likelihood_M1, res_1_0_3.x, method="Nelder-Mead", bounds=[(-20, -18), (0, 1), (1, 6), (-0.3, 0.3), (-15, 0)])

logL0 = log_likelihood_M0(fit_0.x)
logL1 = log_likelihood_M1(fit_1.x)
BIC0 = -2*logL0 + 4*np.log(len(mB_kept))
BIC1 = -2*logL1 + 5*np.log(len(mB_kept))

print("BIC0 - BIC1 =", round(BIC0 - BIC1, 3))

**ANSWER:**

I chose a uniform prior on gamma from -0.3 to 0.3. This is the same prior that i used earlier in the assignment for gamma and that I determined using the weak relationship I saw in the plot of the Hubble residuals against the log_hostmass. Since the relationship did not appear very strong, I reasoned a bound of this magnitude. The corner plot in the previous part of this problem seems to suggest that perhaps this prior should be tighter, but I do not want to use the same data twice in this manner as this is not how a prior should be selected. Therefore, I stayed with the uniform range from -0.3 to 0.3. We can see that this gives a value of 2lnB_1,0 of 1.215 which is weak evidence favoring M1 while using a wider prior of -0.6 to 0.6 gives a value of -0.171 which slightly favors M0 but is inconclusive. This is exactly what we expect as the Occam penalty is increased for M1 with an increased prior width.

The BIC0 - BIC1 calculation gives 0.838 which means very weak evidence in favor of M1 on the scale we used on Day 8. This agrees with the weak evidence obtained from the Bayes factor for a -0.3 to 0.3 uniform prior. The fitted gamma is -0.048 +/- 0.019 mag which agrees very well with the published value of -0.06 despite the weak evidence given by the tests. Since we have an uncertainty of 0.019 mag, and 0.06/0.019 is approximately 3.15 sigma, the 250 supernovae could have detected a step that size at a reasonable certainty of 3 sigma.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here
rng = np.random.default_rng(999)
M_true = -19.3
alpha_true = 0.13
beta_true = 2.76
gamma_true = -0.06
sigma_int_true = 0.11
log_sigma_int_true = np.log(sigma_int_true)

n_sim = 200
sigma_mu_pec = (5 / np.log(10)) * 250 / (299792 * z)

model_true = (M_true + mu - alpha_true * x1 + beta_true * c + gamma_true * Theta)
v_true = np.array([1, alpha_true, -beta_true])
vT_C_v_true = np.zeros(len(mB))
for i in range(len(mB)):
    vT_C_v_true[i] = v_true @ Cov[i] @ v_true
total_var_true = vT_C_v_true + sigma_int_true**2 + sigma_mu_pec**2

def neg_log_likelihood(theta, mB_obs):
    M, alpha, beta, gamma, log_sigma_int = theta
    sigma_int = np.exp(log_sigma_int)

    model_mB = M + mu - alpha * x1 + beta * c + gamma * Theta
    res = mB_obs - model_mB

    v = np.array([1, alpha, -beta])
    vT_C_v = np.zeros(len(mB_obs))
    for i in range(len(mB_obs)):
        vT_C_v[i] = v @ Cov[i] @ v
    total_var = vT_C_v + sigma_int**2 + sigma_mu_pec**2

    return 0.5 * np.sum(np.log(total_var) + res**2 / total_var)
    
count_M = 0
count_alpha = 0
count_beta = 0
count_gamma = 0
count_log_sigma_int = 0

for i in range(n_sim):
    mB_sim = model_true + rng.normal(0, np.sqrt(total_var_true), size = len(mB))
    res = minimize(neg_log_likelihood, [-19.5, 0.15, 3, 0, np.log(0.1)], args = (mB_sim), method = 'Nelder-Mead')

    n = len(res.x)
    h = 1e-4
    H = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            ei = np.zeros(n)
            ej = np.zeros(n)
            ei[i] = h
            ej[j] = h

            H[i, j] = (
                neg_log_likelihood(res.x + ei + ej, mB_sim)
                - neg_log_likelihood(res.x + ei - ej, mB_sim)
                - neg_log_likelihood(res.x - ei + ej, mB_sim)
                + neg_log_likelihood(res.x - ei - ej, mB_sim)
            ) / (4 * h**2)

    err_sim = np.sqrt(np.diag(np.linalg.inv(H)))

    if abs(res.x[0] - M_true) <= err_sim[0]:
        count_M += 1
    if abs(res.x[1] - alpha_true) <= err_sim[1]:
        count_alpha += 1
    if abs(res.x[2] - beta_true) <= err_sim[2]:
        count_beta += 1
    if abs(res.x[3] - gamma_true) <= err_sim[3]:
        count_gamma += 1
    if abs(res.x[4] - log_sigma_int_true) <= err_sim[4]:
        count_log_sigma_int += 1

M_coverage = count_M / n_sim
alpha_coverage = count_alpha / n_sim
beta_coverage = count_beta / n_sim
gamma_coverage = count_gamma / n_sim
sigma_int_coverage = count_log_sigma_int / n_sim

print("68% coverage:")
print("M =", round(M_coverage, 5))
print("Alpha =", round(alpha_coverage, 5))
print("Beta =", round(beta_coverage, 5))
print("Gamma =", round(gamma_coverage, 5))
print("Sigma_int =", round(sigma_int_coverage, 5))

**ANSWER:**

Before run: I will take the suggestion and generate 200 different fake datasets of supernova using parameters close to the fitted values I obtained above and calculate the maximum likelihood with a Hessian uncertainty estimate for each. I'll then check how often the 68% intervals contain the true values that I generated the data with. A failure here would be if the percentage calculated is substantially less or more than 68%.

After run: The 68% intervals contain the truth 63-73 percent of the time which is completely consistent. If I had simulated more datasets, these values would be closer to 68%.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here

# DEMONSTRATION 1.

d = load("p3_wd_masses")

mass = d['mass_Msun']
med = np.median(mass)
sd = np.std(mass)
keep = np.abs(mass - med) < 3 * sd
m = mass[keep]

plt.figure(figsize = (12, 8))
plt.scatter(np.where(keep)[0], mass[keep], label='Kept')
plt.scatter(np.where(~keep)[0], mass[~keep], label='Clipped')
plt.axhline(med + 3 * sd, color='red', linestyle='--', label='3-sigma clipping limit')
plt.axhline(med - 3 * sd, color='red', linestyle='--')

plt.xlabel('White dwarf number')
plt.ylabel('Mass [Msun]')
plt.legend()
plt.show()

# DEMONSTRATION 2:
def log_prob(p, m):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, s1) + (1 - f) * stats.norm.pdf(m, mu2, s2)))
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size / m.size])
np.random.seed(42)
nwalk, ndim = 32, 5
start = p0 + 1e-3 * np.random.randn(nwalk, ndim)
sampler = emcee.EnsembleSampler(nwalk, ndim, log_prob, args=(m,))
sampler.run_mcmc(start, 3000, progress=False)
chain = sampler.get_chain(discard=500)             
flat = chain.reshape(-1, ndim)
mu1_dat = flat[:, 0]
mu2_dat = flat[:, 2]
def gelman_rubin(x):                                  
    n, mchains = x.shape
    B = n * np.var(x.mean(axis=0), ddof=1)
    W = np.mean(np.var(x, axis=0, ddof=1))
    return np.sqrt(((n - 1) / n * W + B / n) / W)

names = ['mu1', 'sigma1', 'mu2', 'sigma2', 'f']
fig, axs = plt.subplots(5, 1, figsize=(8, 8), sharex=True)
for i, nm in enumerate(names):
    axs[i].plot(chain[:, :, i], lw=0.3, alpha=0.5); axs[i].set_ylabel(nm)
axs[-1].set_xlabel('step'); plt.tight_layout(); plt.show()
best = np.median(flat, axis=0)

mu1_16, mu1_84, mu1_med = np.percentile(mu1_dat, [16, 84, 50])
mu2_16, mu2_84, mu2_med = np.percentile(mu2_dat, [16, 84, 50])
print("Population 1:")
print("Correct uncertainty:", round(mu1_med, 4), "+", round(mu1_84-mu1_med, 4),"/ -", round(mu1_med - mu1_16, 4), "Msun")
print("Ai Reported uncerainty (really the population width):", round(best[1], 4), "Msun")

print("")
print("Population 2:")
print("Correct uncertainty:", round(mu2_med, 4), "+", round(mu2_84-mu2_med, 4),"/ -", round(mu2_med - mu2_16, 4), "Msun")
print("Ai Reported uncerainty (really the population width):", round(best[3], 4), "Msun")

In [ ]:
# DEMONSTRATION 3
mass_err = d['mass_err_Msun'][keep]
def log_prob_err(p, m, err):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    return np.sum(np.log(f * stats.norm.pdf(m, mu1, np.sqrt(s1**2 + err**2)) + (1 - f) * stats.norm.pdf(m, mu2, np.sqrt(s2**2 + err**2))))
np.random.seed(42)
start_err = p0 + 1e-3 * np.random.randn(nwalk, ndim)

sampler_err = emcee.EnsembleSampler(nwalk, ndim, log_prob_err, args=(m, mass_err))
sampler_err.run_mcmc(start_err, 3000, progress=False)
chain_err = sampler_err.get_chain(discard=500)             
flat_err = chain_err.reshape(-1, ndim)
best_err = np.median(flat_err, axis=0)

print("Original Fit Results")
print("mu1:", round(best[0], 3))
print("sigma1:", round(best[1], 3))
print("mu2:", round(best[2], 3))
print("sigma2:", round(best[3], 3))
print("f:", round(best[4], 3))
print("")
print("Error-Included Fit Results")
print("mu1:", round(best_err[0], 3))
print("sigma1:", round(best_err[1], 3))
print("mu2:", round(best_err[2], 3))
print("sigma2:", round(best_err[3], 3))
print("f:", round(best_err[4], 3))

**REFEREE REPORT:**

FLAW 1: The AI solution claimed to do a 3-sigma clipping in order to get rid of data points in which spectroscopic mass determinations failed, but provided no justification for why this 3-sigma clipping threshold was used. The first demonstration shows a plot of the kept and clipped points, and it is clear that there is no real distinction between these points and those below that makes them clearly an outlier. In fact, there are many points that are very close but just under this 3-sigma threshold, and so what justification is therefore for those points being kept if the 3sigma ones were removed? This is a problem because it takes away data from the higher mass population and so will bias that population towards a lower mass. The correct treatment is to not clip points unless there is other evidence that a measurement is reliable, or other evidence for a certain level of sigma clipping. The mixture model could also be adapted to take in all points and fit a bad population and good population.

FLAW 2: The AI solution reports the mean of each population with an "uncertainty" that is not truly an uncertainty in the estimated population mean, but rather the standard deviation of that white dwarf population. This is not the same thing, as it describes the populations rather than how precisely the mean was measured which is the true meaning of the uncertainty. This results in an incorrect uncertainty estimate which is obviously a problem. The correct treatment is to estimate uncertainty from the posterior distribution by using the 16th-84th percentiles of the samples from MCMC. I perform this treatment and compare in Demonstration 2, where the AI error and its effect becomes clear as these values are completely different.

FLAW 3: The AI solution ignores the given measurement errors in the masses and instead treats the masses as exact. Demonstration 3 compares the fitted parameters you get when considering and not considering these measurement errors. We can see in the Demonstration that including the errors does not appreciably change the fit results for most of the parameters, but it does have a big impact on the population 1 width, as the measurement errors are comparable to sigma 1 and so sigma 1 is overestimated if the errors are not considered. The same thing cannot be seen for sigma2 as sigma2 is larger than the scale of the measurement errors. The correct treatment is therefore the one I took in the demonstration where the errors are considered.

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

I didn't use any and everything took very long as a result :)

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.